# Module 9: RAG & LLM Applications
## 07: Conversation Memory & Context Management

### Theoretical Overview
Since LLM APIs are stateless HTTP transactions, conversational multi-turn assistants must maintain dialogue history. Memory architectures include:
- **Full History Buffer**: Appends all prior turns. High token consumption and risk of exceeding context window.
- **Sliding Window Buffer**: Retains only the last $K$ interaction turns. Fixed memory footprint.
- **Summary Buffer**: Periodically uses the LLM to compress older turns into an evolving narrative summary.
- **Entity / Graph Memory**: Extracts and tracks persistent attributes of people, organizations, and topics.

In [ ]:
class SlidingWindowMemory:
    """Retains the most recent K dialogue turns."""
    def __init__(self, max_turns: int = 3):
        self.max_turns = max_turns
        self.buffer = []
        
    def add_turn(self, user_msg: str, assistant_msg: str):
        self.buffer.append({"role": "user", "content": user_msg})
        self.buffer.append({"role": "assistant", "content": assistant_msg})
        # Keep only the last max_turns * 2 messages
        if len(self.buffer) > self.max_turns * 2:
            self.buffer = self.buffer[-(self.max_turns * 2):]
            
    def get_messages(self) -> List[Dict[str, str]]:
        return list(self.buffer)

mem = SlidingWindowMemory(max_turns=2)
mem.add_turn("Hi, I'm Alex from Boston.", "Hello Alex! How can I help you today?")
mem.add_turn("What is my name and city?", "You are Alex from Boston.")
mem.add_turn("Tell me a fact about space.", "A day on Venus is longer than a year on Venus.")

print("Retained Memory Window (Last 2 turns):")
for msg in mem.get_messages():
    print(f"[{msg['role'].upper()}]: {msg['content']}")

### Summary Memory Evolution

In [ ]:
class SummaryMemorySimulator:
    def __init__(self):
        self.running_summary = ""
        
    def update_summary(self, old_dialogue: str):
        # Simulates LLM summarizing older dialogue
        self.running_summary += f" [User previously discussed: {old_dialogue[:40]}...]"
        
    def get_context(self) -> str:
        return f"Historical Context Summary: {self.running_summary}"

summarizer = SummaryMemorySimulator()
summarizer.update_summary("User ordered 10 widgets for project Apollo")
summarizer.update_summary("User modified delivery address to Building 4")
print(summarizer.get_context())